In [1]:
import os
import pandas as pd

In [7]:
# Define data directory path
data_path = "C:/Users/gopik/olist_project/data/"

In [8]:

# List and print all files to confirm they are accessible
files = os.listdir(data_path)
print("Files found in folder:", files)

Files found in folder: ['olist_customers_dataset.csv', 'olist_geolocation_dataset.csv', 'olist_orders_dataset.csv', 'olist_order_items_dataset.csv', 'olist_order_payments_dataset.csv', 'olist_order_reviews_dataset.csv', 'olist_products_dataset.csv', 'olist_sellers_dataset.csv', 'product_category_name_translation.csv']


In [9]:

# Load core dataset tables into pandas DataFrames
orders_df = pd.read_csv(data_path + "olist_orders_dataset.csv")
order_items_df = pd.read_csv(data_path + "olist_order_items_dataset.csv")
products_df = pd.read_csv(data_path + "olist_products_dataset.csv")
customers_df = pd.read_csv(data_path + "olist_customers_dataset.csv")
category_translation_df = pd.read_csv(
    data_path + "product_category_name_translation.csv"
)

In [10]:

# Display dataset dimensions (rows, columns)
print("\n--- Dataset Dimensions ---")
print("Orders:", orders_df.shape)
print("Order Items:", order_items_df.shape)
print("Products:", products_df.shape)
print("Customers:", customers_df.shape)


--- Dataset Dimensions ---
Orders: (99441, 8)
Order Items: (112650, 7)
Products: (32951, 9)
Customers: (99441, 5)


In [11]:
# List of columns that represent dates/timestamps
date_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

# Convert date columns to proper pandas datetime types
for col in date_cols:
    orders_df[col] = pd.to_datetime(orders_df[col])

# Check missing values count in the orders table
print("Missing values in Orders table:")
print(orders_df.isnull().sum())

# Drop rows missing essential purchase timestamps
orders_df = orders_df.dropna(subset=["order_purchase_timestamp"])
print(
    "\nCleaned Orders dataset row count:",
    len(orders_df),
)

Missing values in Orders table:
order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64

Cleaned Orders dataset row count: 99441


In [12]:
pip install pymysql sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [14]:
from sqlalchemy import create_engine, text

USER = "root"
PASSWORD = "Gops@3031"  # Replace with your actual password
HOST = "127.0.0.1"
PORT = 3306

try:
    engine = create_engine(f"mysql+pymysql://{USER}:{PASSWORD}@{HOST}:{PORT}")
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 'Connection Successful!'"))
        print(result.fetchone()[0])
except Exception as e:
    print("Connection Failed Error:")
    print(e)

Connection Failed Error:
(pymysql.err.OperationalError) (2003, "Can't connect to MySQL server on '3031@127.0.0.1' ([Errno 11003] getaddrinfo failed)")
(Background on this error at: https://sqlalche.me/e/20/e3q8)


In [15]:
import pymysql
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# Define variables cleanly
USER = "root"
PASSWORD = "Gops@3031"   # Replace with your actual password
HOST = "127.0.0.1"
PORT = 3306

# Create an explicit connection URL object
connection_url = URL.create(
    drivername="mysql+pymysql",
    username=USER,
    password=PASSWORD,
    host=HOST,
    port=PORT,
)

try:
    engine = create_engine(connection_url)
    with engine.connect() as conn:
        result = conn.execute(text("SELECT 'Connection Successful!'"))
        print(result.fetchone()[0])
except Exception as e:
    print("Connection Failed Error:")
    print(e)

Connection Successful!


In [16]:
import pymysql
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# 1. Database Credentials
USER = "root"
PASSWORD = "*******"   # Replace with your actual password
HOST = "127.0.0.1"
PORT = 3306
DATABASE = "olist_ecommerce"

# 2. Server URL (to create database)
server_url = URL.create(
    drivername="mysql+pymysql",
    username=USER,
    password=PASSWORD,
    host=HOST,
    port=PORT,
)

# Connect to MySQL and create database
server_engine = create_engine(server_url)
with server_engine.connect() as conn:
    conn.execute(text(f"CREATE DATABASE IF NOT EXISTS {DATABASE};"))
print(f"Database '{DATABASE}' created/verified successfully!")

# 3. Database URL (to load tables into olist_ecommerce)
db_url = URL.create(
    drivername="mysql+pymysql",
    username=USER,
    password=PASSWORD,
    host=HOST,
    port=PORT,
    database=DATABASE,
)

db_engine = create_engine(db_url)

# 4. Upload DataFrames to MySQL
orders_df.to_sql("orders", con=db_engine, if_exists="replace", index=False)
order_items_df.to_sql(
    "order_items", con=db_engine, if_exists="replace", index=False
)
products_df.to_sql("products", con=db_engine, if_exists="replace", index=False)
customers_df.to_sql(
    "customers", con=db_engine, if_exists="replace", index=False
)
category_translation_df.to_sql(
    "product_category_translation",
    con=db_engine,
    if_exists="replace",
    index=False,
)

print("All tables successfully loaded into MySQL database!")

Database 'olist_ecommerce' created/verified successfully!
All tables successfully loaded into MySQL database!
